# Exercise 2: Performing a capacity expansion model

## Tasks:

* Prepare a capacity expansion plan with solar and wind generators.
* Include capacity expansion of BESS.
* Calculate $CO_2$ emissions using the function provided.
* Prepare a capacity expansion when reducing $CO_2$ emissions by 20%.

In [2]:
lesson_number = 2
print(f'Exercise {lesson_number}')

Exercise 2


In [3]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = True #@param {type:"boolean"}

import os
if 'lesson_number' not in locals(): lesson_number = 4

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/energylab-emw-2026'
    lesson_folder = f'exercise-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

Mounted at /content/drive
Directory '/content/drive/MyDrive/energylab-emw-2026' already exists.
Directory '/content/drive/MyDrive/energylab-emw-2026/exercise-2' created.
Current working directory: /content/drive/MyDrive/energylab-emw-2026/exercise-2


In [4]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = True #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install pypsa
  !pip install pypsa[excel]
  !pip install folium mapclassify cartopy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.0/369.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.6/184.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.2/920.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 82.3 MB/s eta 0:00:00


In [5]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = True #@param {type:"boolean"}

import urllib.request
import os

if DOWNLOAD_FILE:
    url = "https://raw.githubusercontent.com/PriyeshGosai/energylab-emw-2026/main/exercise_2.xlsx"
    file_name = url.split('/')[-1]  # Extract filename from URL
    filename= file_name
    urllib.request.urlretrieve(url, filename)
    print(f"File downloaded successfully: {os.path.abspath(filename)}")

else:
    print("Skipping file download.")

File downloaded successfully: /content/drive/MyDrive/energylab-emw-2026/exercise-2/exercise_2.xlsx


## Custom Functions

In [6]:
def calculate_generator_co2(n):
    """
    Calculate total and per-generator CO2 emissions from actual power output.

    Parameters
    ----------
    n : pypsa.Network
        The network object

    Returns
    -------
    total_co2 : float
        Total CO2 emissions in tonnes
    df_co2 : pd.DataFrame
        DataFrame with CO2 per generator including efficiency, emissions factor, and p_nom
    """
    import pandas as pd

    gen_power = n.generators.dynamic.p  # Power output over time
    gen_co2_by_generator = {}

    for gen_name in gen_power.columns:
        carrier = n.generators.static.loc[gen_name, 'carrier']
        co2_rate = n.carriers.static.loc[carrier, 'co2_emissions']
        efficiency = n.generators.static.loc[gen_name, 'efficiency']
        p_nom = n.generators.static.loc[gen_name, 'p_nom']

        # Sum actual power output and multiply by CO2 rate, adjusted for efficiency
        gen_co2 = gen_power[gen_name].sum() * co2_rate / efficiency

        gen_co2_by_generator[gen_name] = {
            'co2_emissions': gen_co2,
            'efficiency': efficiency,
            'emissions_factor': co2_rate,
            'p_nom': p_nom
        }

    # Convert to DataFrame
    df_co2 = pd.DataFrame(gen_co2_by_generator).T
    df_co2 = df_co2.sort_values('co2_emissions', ascending=False)

    total_co2 = df_co2['co2_emissions'].sum()

    return total_co2, df_co2



## Model

In [ ]:
import pypsa
pypsa.options.api.new_components_api = True

import pandas as pd
pd.options.plotting.backend = 'plotly'

n = pypsa.Network('exercise_2.xlsx')

n.storage_units.static['overnight_cost']




In [ ]:
n.sanitize()
n.optimize()

In [ ]:
total_co2, df_co2 = calculate_generator_co2(n)

In [ ]:
total_co2

In [ ]:
df_co2_coal = df_co2[df_co2.index.str.contains("Coal")]

df_co2_coal['co2_emissions'].plot.bar()

## Results

In [ ]:
n.generators.static.p_nom_opt.plot.barh()

In [ ]:
n.storage_units.static.p_nom_opt.plot.barh()

In [ ]:
s = n.statistics

df = s.curtailment(groupby = False, groupby_time = False, carrier = ['solar','wind']).T

df.columns = df.columns.get_level_values("name")

df.plot()

In [ ]:
s.curtailment(groupby = False, carrier = ['solar','wind'])


In [ ]:
n.generators.dynamic.p.plot.area(stacked = True)

In [ ]:
n.storage_units.dynamic.state_of_charge.plot(title="Battery State of Charge",
                                            labels={
                                                "value": "State of Charge (MWh)",
                                                "index": "Time"
                                            })

In [ ]:
import pandas as pd

p_dispatch = n.storage_units.dynamic.p_dispatch.copy()
p_store = (n.storage_units.dynamic.p_store*-1).copy()

# prefix column names to distinguish storage dispatch/store from generators
p_dispatch.columns = "Discharge " + p_dispatch.columns
p_store.columns = "Store " + p_store.columns

p_all = pd.concat(
    [p_store, n.generators.dynamic.p, p_dispatch],
    axis=1
)

p_all.plot.area()

In [ ]:
n_example = pypsa.examples.ac_dc_meshed()

In [ ]:
n_example.global_constraints.static